In [9]:
import cv2
import numpy as np

# Open the laptop camera
cap = cv2.VideoCapture(0)

# Get the frame width, height, and frames per second information
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = 30

# Define the codec and create a VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # You can use other codecs like 'XVID' or 'MJPG'
out = cv2.VideoWriter('output_track_fast.mp4', fourcc, fps, (frame_width, frame_height))

# Parameters for ShiTomasi corner detection
feature_params = dict(maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)

# Parameters for Lucas-Kanade optical flow
lk_params = dict(winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# Create some random colors for visualization
color = np.random.randint(0, 255, (100, 3))

# Take the first frame and find corners in it
ret, old_frame = cap.read()
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

# Create a mask image for drawing purposes
mask = np.zeros_like(old_frame)

# Dictionary to store the appearance frame for each point
appearance_frames = {}

# while True:
for _ in range(300):    # with 30 FPS save video for 10 seconds 
    ret, frame = cap.read()
    if not ret:
        print('No frames grabbed!')
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Calculate optical flow using Lucas-Kanade
    p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

    # Select good points
    if p1 is not None:
        good_new = p1[st == 1]
        good_old = p0[st == 1]

    # Draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel()
        c, d = old.ravel()

        # Update appearance frames for each point
        if (a, b) not in appearance_frames:
            appearance_frames[(a, b)] = 0

        # Draw the tracks with fading effect
        alpha = max(0, 1.0 - appearance_frames[(a, b)] / 60.0)
        color_alpha = tuple(int(alpha * c) for c in color[i])
        mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color_alpha, 2)
        frame = cv2.circle(frame, (int(a), int(b)), 5, color_alpha, -1)

        # Increment appearance frame counter
        appearance_frames[(a, b)] += 1

    # Combine the frame with the movement path mask
    img = cv2.add(frame, mask)

    # Write the frame to the output video file
    out.write(img)

    # Display the result
    cv2.imshow('Optical Flow', img)

    # Break the loop if the 'Esc' key is pressed
    if cv2.waitKey(30) & 0xFF == 27:
        break

    # Update the previous frame and previous points
    old_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)

# Release the capture object, release the video writer, and close windows
cap.release()
out.release()
cv2.destroyAllWindows()

[ WARN:0@1109.328] global /croot/opencv-suite_1676452025216/work/modules/videoio/src/cap_gstreamer.cpp (862) isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created
